In [45]:
from docling.document_converter import DocumentConverter

converter = DocumentConverter()
document_source = "https://arxiv.org/pdf/1706.03762"
document = converter.convert(source=document_source).document

In [46]:
from docling_core.transforms.chunker.tokenizer.base import BaseTokenizer
from docling_core.transforms.chunker.tokenizer.huggingface import HuggingFaceTokenizer
from transformers import AutoTokenizer

EMBED_MODEL_ID = "sentence-transformers/all-MiniLM-L6-v2"

tokenizer: BaseTokenizer = HuggingFaceTokenizer(
    tokenizer=AutoTokenizer.from_pretrained(EMBED_MODEL_ID),
)

In [47]:
print(f"{tokenizer.get_max_tokens()=}")

tokenizer.get_max_tokens()=256


In [48]:
from typing import Iterable, Optional

from docling_core.transforms.chunker.base import BaseChunk
from docling_core.transforms.chunker.hierarchical_chunker import DocChunk
from docling_core.types.doc.labels import DocItemLabel
from rich.console import Console
from rich.panel import Panel


console = Console(width=200)

In [ ]:
from docling_core.transforms.chunker.hierarchical_chunker import (
    ChunkingDocSerializer,
    ChunkingSerializerProvider,
    HierarchicalChunker,
)
from docling_core.transforms.serializer.markdown import MarkdownTableSerializer, MarkdownParams, MarkdownTextSerializer


class MDTableSerializerProvider(ChunkingSerializerProvider):
    def get_serializer(self, doc):
        return ChunkingDocSerializer(
            doc=doc,
            params=MarkdownParams(image_placeholder="<!-- image -->"),
            table_serializer=MarkdownTableSerializer(),  # configuring a different table serializer
            text_serializer=MarkdownTextSerializer(),
        )


chunker = HierarchicalChunker(
    serializer_provider=MDTableSerializerProvider(),
    merge_list_items=True,
)

chunk_iter = chunker.chunk(dl_doc=document)


def find_n_th_chunk_with_label(
    iter: Iterable[BaseChunk], n: int, label: DocItemLabel
) -> Optional[DocChunk]:
    num_found = -1
    for i, chunk in enumerate(iter):
        doc_chunk = DocChunk.model_validate(chunk)
        for it in doc_chunk.meta.doc_items:
            if it.label == label:
                num_found += 1
                if num_found == n:
                    return i, chunk
    return None, None


def print_chunk(chunks, chunk_pos):
    chunk = chunks[chunk_pos]
    ctx_text = chunker.contextualize(chunk=chunk)
    num_tokens = tokenizer.count_tokens(text=ctx_text)
    doc_items_refs = [it.self_ref for it in chunk.meta.doc_items]
    title = f"{chunk_pos=} {num_tokens=} {doc_items_refs=}"
    console.print(Panel(ctx_text, title=title))


chunks = list(chunk_iter)
i, chunk = find_n_th_chunk_with_label(chunks, n=0, label=DocItemLabel.TABLE)
if i is not None:
    print_chunk(
        chunks=chunks,
        chunk_pos=i,
    )

╭────────────────────────────────────────────────────────────── chunk_pos=98 num_tokens=293 doc_items_refs=['#/texts/122', '#/tables/0'] ──────────────────────────────────────────────────────────────╮
│ 3.4 Embeddings and Softmax                                                                                                                                                                           │
│ Table 1: Maximum path lengths, per-layer complexity and minimum number of sequential operations for different layer types. n is the sequence length, d is the representation dimension, k is the     │
│ kernel size of convolutions and r the size of the neighborhood in restricted self-attention.                                                                                                         │
│                                                                                                                                                                                                      │
│ | Layer Type                  | Complexity per Layer   | Sequential Operations   | Maximum Path Length   |                                                                                           │
│ |-----------------------------|------------------------|-------------------------|-----------------------|                                                                                           │
│ | Self-Attention              | O ( n 2 · d )          | O (1)                   | O (1)                 |                                                                                           │
│ | Recurrent                   | O ( n · d 2 )          | O ( n )                 | O ( n )               |                                                                                           │
│ | Convolutional               | O ( k · n · d 2 )      | O (1)                   | O ( log k ( n ))      |                                                                                           │
│ | Self-Attention (restricted) | O ( r · n · d )        | O (1)                   | O ( n/r )             |                                                                                           │
╰──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [50]:
def _print_chunk(chunk, chunk_pos):
    ctx_text = chunker.contextualize(chunk=chunk)
    num_tokens = tokenizer.count_tokens(text=ctx_text)
    doc_items_refs = [it.self_ref for it in chunk.meta.doc_items]
    title = f"{chunk_pos=} {num_tokens=} {doc_items_refs=}"
    console.print(Panel(ctx_text, title=title))

chunk_id = 0
for chunk in chunker.chunk(dl_doc=document):
    _print_chunk(chunk, chunk_id)
    chunk_id += 1

╭─────────────────────────────────────────────────────────────────────── chunk_pos=0 num_tokens=32 doc_items_refs=['#/texts/1'] ───────────────────────────────────────────────────────────────────────╮
│ Provided proper attribution is provided, Google hereby grants permission to reproduce the tables and figures in this paper solely for use in journalistic or scholarly works.                        │
╰──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────────────────────────────── chunk_pos=1 num_tokens=20 doc_items_refs=['#/texts/3'] ───────────────────────────────────────────────────────────────────────╮
│ Attention Is All You Need                                                                                                                                                                            │
│ Ashish Vaswani ∗ Google Brain avaswani@google.com                                                                                                                                                    │
╰──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────────────────────────────── chunk_pos=2 num_tokens=19 doc_items_refs=['#/texts/4'] ───────────────────────────────────────────────────────────────────────╮
│ Attention Is All You Need                                                                                                                                                                            │
│ Noam Shazeer ∗ Google Brain noam@google.com                                                                                                                                                          │
╰──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────────────────────────────── chunk_pos=3 num_tokens=17 doc_items_refs=['#/texts/5'] ───────────────────────────────────────────────────────────────────────╮
│ Attention Is All You Need                                                                                                                                                                            │
│ Llion Jones ∗ Google Research llion@google.com                                                                                                                                                       │
╰──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────────────────────────────── chunk_pos=4 num_tokens=18 doc_items_refs=['#/texts/6'] ───────────────────────────────────────────────────────────────────────╮
│ Attention Is All You Need                                                                                                                                                                            │
│ Niki Parmar ∗ Google Research nikip@google.com                                                                                                                                                       │
╰──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭───────────────────────────────────────────────────────────── chunk_pos=146 num_tokens=97 doc_items_refs=['#/texts/228', '#/pictures/3'] ─────────────────────────────────────────────────────────────╮
│ Attention Visualizations Input-Input Layer5                                                                                                                                                          │
│ Figure 3: An example of the attention mechanism following long-distance dependencies in the encoder self-attention in layer 5 of 6. Many of the attention heads attend to a distant dependency of    │
│ the verb 'making', completing the phrase 'making...more difficult'. Attentions here shown only for the word 'making'. Different colors represent different heads. Best viewed in color.              │
│                                                                                                                                                                                                      │
│ <!-- image -->                                                                                                                                                                                       │
╰──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────────────────────────────── chunk_pos=147 num_tokens=13 doc_items_refs=['#/texts/296'] ─────────────────────────────────────────────────────────────────────╮
│ Attention Visualizations Input-Input Layer5                                                                                                                                                          │
│ Input-Input Layer5                                                                                                                                                                                   │
╰──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────────────────────── chunk_pos=148 num_tokens=78 doc_items_refs=['#/texts/297', '#/pictures/4'] ─────────────────────────────────────────────────────────────╮
│ Attention Visualizations Input-Input Layer5                                                                                                                                                          │
│ Figure 4: Two attention heads, also in layer 5 of 6, apparently involved in anaphora resolution. Top: Full attentions for head 5. Bottom: Isolated attentions from just the word 'its' for attention │
│ heads 5 and 6. Note that the attentions are very sharp for this word.                                                                                                                                │
│                                                                                                                                                                                                      │
│ <!-- image -->                                                                                                                                                                                       │
╰──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────────────────────────────── chunk_pos=149 num_tokens=13 doc_items_refs=['#/texts/408'] ─────────────────────────────────────────────────────────────────────╮
│ Attention Visualizations Input-Input Layer5                                                                                                                                                          │
│ Input-Input Layer5                                                                                                                                                                                   │
╰──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────────────────────── chunk_pos=150 num_tokens=70 doc_items_refs=['#/texts/409', '#/pictures/5'] ─────────────────────────────────────────────────────────────╮
│ Attention Visualizations Input-Input Layer5                                                                                                                                                          │
│ Figure 5: Many of the attention heads exhibit behaviour that seems related to the structure of the sentence. We give two such examples above, from two different heads from the encoder              │
│ self-attention at layer 5 of 6. The heads clearly learned to perform different tasks.                                                                                                                │
│                                                                                                                                                                                                      │
│ <!-- image -->                                                                                                                                                                                       │
╰──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [ ]:
from docling_core.transforms.serializer.markdown import MarkdownParams


class ImgPlaceholderSerializerProvider(ChunkingSerializerProvider):
    def get_serializer(self, doc):
        return ChunkingDocSerializer(
            doc=doc,
            params=MarkdownParams(
                image_placeholder="<!-- image -->",
            ),
        )


chunker = HierarchicalChunker(
    serializer_provider=ImgPlaceholderSerializerProvider(),
    merge_list_items=True,
)

chunk_iter = chunker.chunk(dl_doc=document)

chunks = list(chunk_iter)
i, chunk = find_n_th_chunk_with_label(chunks, n=0, label=DocItemLabel.PICTURE)
print_chunk(
    chunks=chunks,
    chunk_pos=i,
)

╭────────────────────────────────────────────────────────────── chunk_pos=29 num_tokens=25 doc_items_refs=['#/texts/34', '#/pictures/0'] ──────────────────────────────────────────────────────────────╮
│ Related Work                                                                                                                                                                                         │
│ Figure 1: System architecture: Simplified sketch of document question-answering pipeline.                                                                                                            │
│                                                                                                                                                                                                      │
│ <!-- image -->                                                                                                                                                                                       │
╰──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯